# ITU-T RAG — Quickstart Notebook

**Open Telco RAG**

This notebook walks through the Open Telco RAG pipeline for the **ITU-T** corpus —
recommendations across 14 active telecom series.

By the end of this notebook you will have:
- Ingested ITU-T recommendations into a local ChromaDB
- Queried AI/ML framework specifications (Y.3172, Y.3173, Y.3176)
- Asked questions about intelligence levels and ML in future networks
- Evaluated retrieval quality with the evaluation metrics

---

### Prerequisites

```bash
git clone https://github.com/gsma-open-telco-ai/open-telco-rag
cd open-telco-rag
pip install -e .
```

Download ITU-T recommendations:
```bash
python scripts/download_itu.py --output data/itu --series G E Q H X Y M L P K J F T V
```

Set your API keys in a `.env` file:
```
# For answer generation (choose one or both)
ANTHROPIC_API_KEY=sk-ant-...        # Claude models direct
OPENROUTER_API_KEY=sk-or-...        # Any model via OpenRouter
```

## 1. Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from collections import Counter
import statistics

load_dotenv()

DATA_DIR   = Path("data/itu")
DB_PATH    = "chroma_db"
MODEL_KEY  = "minilm"
COLLECTION = f"itu_{MODEL_KEY}"

print(f"Data directory : {DATA_DIR}")
print(f"ChromaDB path  : {DB_PATH}")
print(f"Embedding model: {MODEL_KEY}")
print(f"Collection     : {COLLECTION}")

## 2. Check available recommendations

All 14 active telecom series are downloaded to `data/itu/` in series subfolders.
Delete the 4 duplicate root-level PDFs before ingesting:
```bash
del data\itu\T-REC-Y*.pdf
```

In [ ]:
if DATA_DIR.exists():
    series_dirs = [d for d in DATA_DIR.iterdir() if d.is_dir()]
    pdf_files   = list(DATA_DIR.rglob("*.pdf"))
    root_pdfs   = [f for f in DATA_DIR.iterdir() if f.suffix == ".pdf"]
    print(f"Series folders   : {len(series_dirs)}")
    print(f"Total PDFs       : {len(pdf_files):,}")
    if root_pdfs:
        print(f"\nRoot PDFs (duplicates — delete before ingesting):")
        for f in sorted(root_pdfs):
            print(f"  {f.name}")
    print("\nSeries coverage:")
    for d in sorted(series_dirs):
        n = len(list(d.rglob("*.pdf")))
        print(f"  {d.name:<4} {n:>6,} PDFs")
else:
    print(f"[INFO] {DATA_DIR} not found.")
    print("Download: python scripts/download_itu.py --output data/itu")

## 3. Ingest into ChromaDB

The ITU-T ingest pipeline:
1. Loads PDF recommendations from all 14 series subfolders
2. Extracts metadata from filenames: series, recommendation number,
   approval date, status (In force / Superseded / Provisional)
3. Applies paragraph-based chunking with overlap
4. Embeds and stores with series letter in metadata

In [ ]:
from open_telco_rag.sources.itu import ITUSource
from open_telco_rag.embedders.sentense_transformers import SentenceTransformerEmbedder
from open_telco_rag.stores.chroma import ChromaStore
from open_telco_rag.core.pipeline import ingest

source   = ITUSource()
embedder = SentenceTransformerEmbedder(model_key=MODEL_KEY)
store    = ChromaStore(path=DB_PATH, collection_name=COLLECTION)

print(f"Starting ingest — {store.count()} chunks already in collection")

if DATA_DIR.exists():
    stats = ingest(
        source     = source,
        store      = store,
        embedder   = embedder,
        data_dir   = str(DATA_DIR),
        chunk_size = 1024,
        overlap    = 200,
        resume     = True,
    )
    print(f"\nIngest complete")
    print(f"  Files processed : {stats.files_processed}")
    print(f"  Chunks stored   : {stats.chunks_stored}")
    print(f"  Skipped         : {stats.skipped_files}")
    print(f"  Errors          : {stats.error_files}")
print(f"  Total in DB     : {store.count():,}")

## 4. Explore the Collection

In [ ]:
import chromadb

client     = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_collection(COLLECTION)
sample     = collection.get(limit=2000, include=["documents", "metadatas"])

series_counts = Counter(m.get("series", "?") for m in sample["metadatas"])
lengths       = [len(d) for d in sample["documents"]]

SERIES_DESC = {
    "Y": "NGN, cloud, AI/ML, IoT",
    "G": "Transmission systems and media",
    "X": "Data networks and security",
    "Q": "Switching and signalling",
    "H": "Audiovisual and multimedia",
    "E": "Network operation, QoS",
    "M": "Telecommunication management",
    "L": "Environment and cable",
}

print(f"Collection: {COLLECTION}")
print(f"Total chunks: {collection.count():,}\n")
print(f"Series distribution (sample of 2,000):")
print(f"{'Series':>8} {'Chunks':>8}  Description")
print("-" * 55)
for series, count in series_counts.most_common(12):
    desc = SERIES_DESC.get(series, "")
    print(f"{series:>8} {count:>8}  {desc}")

print(f"\nChunk length statistics (chars):")
print(f"  Mean   : {statistics.mean(lengths):.0f}")
print(f"  Median : {statistics.median(lengths):.0f}")

## 5. Semantic Search

The Y-series recommendations Y.3172, Y.3173, and Y.3176 define the
international AI/ML framework for future networks — directly relevant
to the Open Telco AI community.

In [ ]:
from open_telco_rag.core.pipeline import search

QUERY = "What does ITU-T Y.3172 define about machine learning in future networks?"
TOP_K = 5

results = search(
    QUERY,
    store    = store,
    embedder = embedder,
    top_k    = TOP_K,
)

print(f"Query: '{QUERY}'")
print(f"Top-{TOP_K} results:\n")
print(f"{'Rank':<5} {'Similarity':>10} {'Series':<8} {'Rec. number':<15} {'Source'}")
print("-" * 75)

for i, r in enumerate(results, 1):
    sim  = round(1 - r.distance, 3)
    s    = r.metadata.get("series", "")
    rec  = r.metadata.get("rec_number", "")
    src  = r.metadata.get("source", "")[:25]
    print(f"{i:<5} {sim:>10.3f} {s:<8} {rec:<15} {src}")

print(f"\nTop result preview:")
print(results[0].text[:400])

### 5.1 Y-series AI/ML recommendation coverage

Check which Y.3xxx AI/ML recommendations are in the collection.

In [ ]:
ai_queries = [
    "Y.3172 machine learning framework",
    "Y.3173 intelligence levels evaluation",
    "Y.3176 ML-based network operation requirements",
]

print(f"{'Query':<40} {'Best rec.':<15} {'Similarity':>10}")
print("-" * 70)

for q in ai_queries:
    r    = search(q, store=store, embedder=embedder, top_k=3)
    if r:
        sim = 1 - r[0].distance
        rec = r[0].metadata.get("rec_number", "?")
        print(f"{q:<40} {rec:<15} {sim:>10.3f}")
    else:
        print(f"{q:<40} {'no results':<15}")

## 6. Question Answering

In [ ]:
from open_telco_rag.core.pipeline import generate

QUESTION = "What does ITU-T Y.3172 define about machine learning in future networks?"

retrieved = search(QUESTION, store=store, embedder=embedder, top_k=5)

from open_telco_rag.generators.openrouter import OpenRouterGenerator
generator = OpenRouterGenerator(model="meta-llama/llama-3.3-70b-instruct")

response  = generate(QUESTION, retrieved, source_type="itu", generator=generator)

print(f"Question: {QUESTION}")
print(f"{'─'*70}")
print(f"\n{response['answer']}")
print(f"\n{'─'*70}")
print(f"Sources used: {response['sources']}")
print(f"Chunks used : {response['chunks_used']}")

### 6.1 Question bank

In [ ]:
QUESTIONS = [
    ("What does ITU-T Y.3172 define about machine learning in future networks?", "Y"),
    ("What are the intelligence levels defined in ITU-T Y.3173?",               "Y"),
    ("What does Y.3176 specify for ML-based network operation?",                "Y"),
    ("What is the ITU-T machine learning standardisation roadmap?",             "Y"),
    ("How does ITU-T define the functional architecture for AI in networks?",    "Y"),
]

print(f"{'#':<3} {'Expected series':<17} {'Top series':<14} {'Mean sim':>9} {'Question'}")
print("-" * 90)

for i, (question, expected_series) in enumerate(QUESTIONS, 1):
    results  = search(question, store=store, embedder=embedder, top_k=5)
    sims     = [1 - r.distance for r in results]
    mean_sim = sum(sims) / len(sims) if sims else 0
    top_s    = results[0].metadata.get("series", "?") if results else "?"
    hit      = "✓" if top_s == expected_series else "✗"
    q_short  = question[:48] + "..." if len(question) > 48 else question
    print(f"{i:<3} {expected_series:<17} {top_s + ' ' + hit:<14} {mean_sim:>9.3f} {q_short}")

## 7. Evaluation

In [ ]:
from open_telco_rag.evaluation.evaluate import evaluate_corpus
from open_telco_rag.evaluation.questions import load_questions

questions = load_questions("itu")
print(f"Loaded {len(questions)} evaluation questions for ITU-T")
print(f"Questions with reference answers: {sum(1 for q in questions if q.reference_answer)}")

In [ ]:
from open_telco_rag.evaluation._claude import make_llm_call

# RAG generation model (answers questions)
rag_llm_call = make_llm_call("openrouter", "meta-llama/llama-3.3-70b-instruct")

# Metrics evaluation model (scores faithfulness, relevancy, etc.)
llm_call = make_llm_call("openrouter", "anthropic/claude-sonnet-4.5")

result = evaluate_corpus(
    corpus       = "itu",
    store        = store,
    embedder     = embedder,
    questions    = questions,
    model        = MODEL_KEY,
    top_k        = 5,
    delay        = 1.0,
    llm_call     = llm_call,
    rag_llm_call = rag_llm_call,
)


In [ ]:
print(f"\nPer-question results — ITU-T ({MODEL_KEY}, top-k=5)")
print(f"{'#':<3} {'Faith':>6} {'Relev':>6} {'Sim':>6} {'Recall':>7} {'Correct':>8} {'Question'}")
print("-" * 95)

for i, qr in enumerate(result.question_results, 1):
    q_short = qr.question[:45] + "..." if len(qr.question) > 45 else qr.question
    print(
        f"{i:<3} "
        f"{qr.faithfulness:>6.2f} "
        f"{qr.answer_relevancy:>6.2f} "
        f"{qr.mean_similarity:>6.3f} "
        f"{max(0, qr.context_recall):>7.2f} "
        f"{max(0, qr.answer_correctness):>8.2f} "
        f"{q_short}"
    )

print("-" * 95)
print(
    f"{'AVG':<3} "
    f"{result.mean_faithfulness:>6.2f} "
    f"{result.mean_answer_relevancy:>6.2f} "
    f"{result.mean_similarity:>6.3f} "
    f"{max(0, result.mean_context_recall):>7.2f} "
    f"{max(0, result.mean_answer_correctness):>8.2f}"
)


## 8. What the metrics tell us

| Metric | Type | Interpretation |
|--------|------|----------------|
| **Faithfulness** | Generation | Are the claims in the answer grounded in the retrieved chunks? |
| **Answer relevancy** | Generation | Does the answer address what was asked? |
| **Mean similarity** | Retrieval | How confident is the retrieval? Higher = better chunk matches |
| **Context recall** | Retrieval+GT | Were the chunks needed to answer the question retrieved? |
| **Answer correctness** | Generation+GT | Is the answer factually correct vs the reference answer? |
